## README

Custom script to depict the transcription activities of operons and visualizations.

Input:
- Operon boundary in Excel file
- Sequencing depth in tsv files
- Peaks in tsv file
- Mapped PacBio long-read to ref genome in BAM file
- Gene annotation in gff file
- Genome seq in FASTQ file


In [1]:
MOTHER_FOLDER = "/data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis"
ENDS_FOLDER = MOTHER_FOLDER + "/ends_export"
DEPTH_FOLDER = MOTHER_FOLDER + "/seq_depth"
OPERON_FOLDER = MOTHER_FOLDER + "/operon_merge_then_polish_v2_finalize"

ENDS_FILES = {
	"plus_5": ENDS_FOLDER + "/plus.5p.bedgraph",
	"plus_3": ENDS_FOLDER + "/plus.3p.bedgraph",
	"minus_5": ENDS_FOLDER + "/minus.5p.bedgraph",
	"minus_3": ENDS_FOLDER + "/minus.3p.bedgraph",
}

DEPTH_FILES = {
	"plus": DEPTH_FOLDER + "/syn1.PacBio.FLNC.depth_forward.tsv",
	"minus": DEPTH_FOLDER + "/syn1.PacBio.FLNC.depth_reverse.tsv"
}

OPERONS_TSV = OPERON_FOLDER + "/operons.final.tsv"
PEAKS_TSV   = OPERON_FOLDER + "/peaks.tss_tts.tsv"
BAM_PATH    = MOTHER_FOLDER + "/syn1.PacBio.FLNC.sorted.bam"

GENOME_FOLDER = "/data/enguang/CMEODE/TRSC/Transcriptomics/gene_info/"
GFF3_FOLDER = GENOME_FOLDER + "gff3_files"
FASTA_FOLDER = GENOME_FOLDER + "fasta_files"

GFF3_FILE = GFF3_FOLDER + "/syn1.genes.gff3"
FASTA_FILE = FASTA_FOLDER + "/syn1_genome.fasta"

## Cluster Mapped Reads into Isoforms

In [ ]:
# ============================================================
# Patched isoform clustering:
# (1) Read membership uses CORE overlap criteria (your rules)
# (2) Optional post-merge to collapse micro-variant ends
# ============================================================

from __future__ import annotations
import os
from dataclasses import dataclass
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import pysam


# ----------------------------
# Inputs
# ----------------------------
OPERONS_TSV = OPERON_FOLDER + "/operons.final.tsv"
PEAKS_TSV   = OPERON_FOLDER + "/peaks.tss_tts.tsv"
BAM_PATH    = MOTHER_FOLDER + "/syn1.PacBio.FLNC.sorted.bam"

CLUSTERED_ISOFORMS_FOLDER = MOTHER_FOLDER + "/isoform_clustering"
Path(CLUSTERED_ISOFORMS_FOLDER).mkdir(parents=True, exist_ok=True)
OUT_ISOFORMS_TSV    = CLUSTERED_ISOFORMS_FOLDER + "/isoform_clusters.tsv"
OUT_MEMBERSHIP_TSV  = CLUSTERED_ISOFORMS_FOLDER + "/isoform_membership.tsv"   # set to None to disable membership output

OUT_ISOFORMS_TSV    = "isoform_clusters.tsv"
OUT_MEMBERSHIP_TSV  = None  # set to filename if you want membership


# ----------------------------
# Membership rule (copied from your operon finalization)
# ----------------------------
FETCH_FLANK = 1000
MIN_CORE_OVERLAP_BP = 50
MIN_CORE_OVERLAP_FRAC = 0.50

# Which interval is the "core"?
# Strongly recommended: polished, because it's your anchored core used in finalization.
CORE_START_COL = "start0_polished"
CORE_END_COL   = "end0_polished"

# If you prefer merged core:
# CORE_START_COL = "start0_merged"
# CORE_END_COL   = "end0_merged"


# ----------------------------
# Snapping / binning
# ----------------------------
W5, W3 = 5, 10
BIN5, BIN3 = 20, 30   # <-- bumped to reduce micro-variants

MIN_ISOFORM_READS = 10
MIN_MAPQ = 0
REQUIRE_PRIMARY = True


# ----------------------------
# Optional post-merge (collapse close ends)
# ----------------------------
DO_POST_MERGE = True
MERGE_END_TOL_BP = 15   # merge isoforms with same start_key and end_pos within this tolerance


# ----------------------------
# Helpers
# ----------------------------
def overlap_len(a0: int, a1: int, b0: int, b1: int) -> int:
    return max(0, min(a1, b1) - max(a0, b0))

def read_is_on_strand(read: pysam.AlignedSegment, strand: str) -> bool:
    if strand == "+":
        return not read.is_reverse
    elif strand == "-":
        return read.is_reverse
    raise ValueError(strand)

def aligned_ends_pos0(read: pysam.AlignedSegment, strand: str) -> Optional[Tuple[int,int]]:
    r0 = read.reference_start
    r1 = read.reference_end
    if r0 is None or r1 is None or r1 <= r0:
        return None
    # aligned coords only (softclips excluded)
    if strand == "+":
        return int(r0), int(r1)  # 5p, 3p
    else:
        return int(r1), int(r0)  # 5p, 3p

def bin_pos(pos0: int, binsz: int) -> int:
    return int((pos0 // binsz) * binsz)

def make_peak_id(peak_type: str, merged_operon_id: int, peak_rank_in_operon: int) -> str:
    pt = str(peak_type).lower()
    if "5" in pt:
        prefix = "5P"
    elif "3" in pt:
        prefix = "3P"
    else:
        prefix = "PX"
    return f"{prefix}_{merged_operon_id}_{int(peak_rank_in_operon)}"

@dataclass
class PeakIndex:
    peaks5: List[Tuple[int,str]]  # (summit_pos0, peak_id)
    peaks3: List[Tuple[int,str]]

def build_peaks_index(peaks_df: pd.DataFrame) -> Dict[Tuple[str,str,int], PeakIndex]:
    idx: Dict[Tuple[str,str,int], PeakIndex] = {}
    for (chrom, strand, opid), g in peaks_df.groupby(["chrom","strand","merged_operon_id"], sort=False):
        p5, p3 = [], []
        for _, r in g.iterrows():
            summit0 = int(r["summit_pos0"])
            pid = make_peak_id(r["peak_type"], int(opid), int(r["peak_rank_in_operon"]))
            pt = str(r["peak_type"]).lower()
            if "5" in pt:
                p5.append((summit0, pid))
            elif "3" in pt:
                p3.append((summit0, pid))
        p5.sort(key=lambda x: x[0])
        p3.sort(key=lambda x: x[0])
        idx[(str(chrom), str(strand), int(opid))] = PeakIndex(peaks5=p5, peaks3=p3)
    return idx

def snap_to_nearest_peak(pos0: int, peaks: List[Tuple[int,str]], win: int) -> Optional[str]:
    best_id, best_d = None, win + 1
    for summit0, pid in peaks:
        d = abs(pos0 - summit0)
        if d <= win and d < best_d:
            best_d = d
            best_id = pid
        if summit0 > pos0 + win:
            break
    return best_id

def isoform_key(spid: Optional[str], epid: Optional[str], pos5p0: int, pos3p0: int) -> Tuple[str,str]:
    start_key = spid if spid is not None else f"RAW5_{bin_pos(pos5p0, BIN5)}"
    end_key   = epid if epid is not None else f"RAW3_{bin_pos(pos3p0, BIN3)}"
    return start_key, end_key

def mad(x: np.ndarray) -> float:
    if x.size <= 1:
        return 0.0
    m = np.median(x)
    return float(np.median(np.abs(x - m)))

def post_merge_isoforms(df: pd.DataFrame, tol_bp: int) -> pd.DataFrame:
    """
    Merge isoforms that share same start_key and have end_pos0 within tol_bp.
    Works within each (chrom,strand,merged_operon_id,start_key).
    """
    if df.empty:
        return df

    out = []
    grp_cols = ["chrom","strand","merged_operon_id","start_key"]
    for _, g in df.sort_values(["chrom","strand","merged_operon_id","start_key","end_pos0"]).groupby(grp_cols, sort=False):
        # greedy merge in sorted end_pos0
        current = None
        for row in g.to_dict("records"):
            if current is None:
                current = row.copy()
                continue
            if abs(row["end_pos0"] - current["end_pos0"]) <= tol_bp:
                # merge into current
                n1 = current["n_reads"]
                n2 = row["n_reads"]
                n  = n1 + n2
                # weighted average for representative end_pos0
                current["end_pos0"] = int(round((current["end_pos0"]*n1 + row["end_pos0"]*n2) / n))
                current["n_reads"] = n
                # keep end_key if both same; if different, mark merged
                if current["end_key"] != row["end_key"]:
                    current["end_key"] = "MERGED_END"
                    current["end_peak_id"] = None
                    current["source"] = "merged"
                # frac_operon_reads recompute later
            else:
                out.append(current)
                current = row.copy()
        if current is not None:
            out.append(current)

    merged = pd.DataFrame(out)
    # recompute frac_operon_reads per operon
    merged["operon_total_reads_used"] = merged.groupby(["chrom","strand","merged_operon_id"])["n_reads"].transform("sum")
    merged["frac_operon_reads"] = merged["n_reads"] / merged["operon_total_reads_used"].clip(lower=1)
    # regenerate isoform_id for stability
    merged["isoform_id"] = merged.apply(
        lambda r: f"op{int(r.merged_operon_id)}__{r.start_key}__{r.end_key}__end{int(r.end_pos0)}",
        axis=1
    )
    return merged


# ----------------------------
# Load
# ----------------------------
operons = pd.read_csv(OPERONS_TSV, sep="\t")
peaks   = pd.read_csv(PEAKS_TSV, sep="\t")

for c in ["chrom","strand","merged_operon_id",CORE_START_COL,CORE_END_COL]:
    if c not in operons.columns:
        raise ValueError(f"operons missing required column: {c}")

peaks_index = build_peaks_index(peaks)

if not os.path.exists(BAM_PATH + ".bai"):
    pysam.index(BAM_PATH)
bam = pysam.AlignmentFile(BAM_PATH, "rb")


# ----------------------------
# Main
# ----------------------------
rows = []
write_membership = OUT_MEMBERSHIP_TSV is not None
mem_rows = []

operons_it = operons.sort_values(["chrom","strand",CORE_START_COL,CORE_END_COL]).reset_index(drop=True)

for _, op in operons_it.iterrows():
    chrom  = str(op["chrom"])
    strand = str(op["strand"])
    opid   = int(op["merged_operon_id"])

    core_s0 = int(op[CORE_START_COL])
    core_e0 = int(op[CORE_END_COL])
    if core_e0 <= core_s0:
        continue

    fetch0 = max(0, core_s0 - FETCH_FLANK)
    fetch1 = core_e0 + FETCH_FLANK

    pk = peaks_index.get((chrom, strand, opid), PeakIndex(peaks5=[], peaks3=[]))

    isoform_groups: Dict[Tuple[str,str], List[Tuple[int,int,str]]] = {}
    operon_reads_used = 0

    try:
        it = bam.fetch(chrom, fetch0, fetch1)
    except ValueError:
        continue

    for read in it:
        if read.is_unmapped:
            continue
        if REQUIRE_PRIMARY and (read.is_secondary or read.is_supplementary):
            continue
        if read.mapping_quality < MIN_MAPQ:
            continue
        if not read_is_on_strand(read, strand):
            continue

        r0, r1 = read.reference_start, read.reference_end
        if r0 is None or r1 is None or r1 <= r0:
            continue

        # ---- CORE-overlap membership (your rule) ----
        ol = overlap_len(int(r0), int(r1), core_s0, core_e0)
        if ol < MIN_CORE_OVERLAP_BP:
            continue
        span = int(r1) - int(r0)
        if (ol / max(1, span)) < MIN_CORE_OVERLAP_FRAC:
            continue

        ends = aligned_ends_pos0(read, strand)
        if ends is None:
            continue
        pos5p0, pos3p0 = ends

        spid = snap_to_nearest_peak(pos5p0, pk.peaks5, W5)
        epid = snap_to_nearest_peak(pos3p0, pk.peaks3, W3)
        sk, ek = isoform_key(spid, epid, pos5p0, pos3p0)

        isoform_groups.setdefault((sk, ek), []).append((pos5p0, pos3p0, read.query_name))
        operon_reads_used += 1

    if operon_reads_used == 0:
        continue

    for (sk, ek), members in isoform_groups.items():
        n = len(members)
        if n < MIN_ISOFORM_READS:
            continue

        pos5 = np.array([m[0] for m in members], dtype=int)
        pos3 = np.array([m[1] for m in members], dtype=int)

        start_pos0 = int(np.median(pos5))
        end_pos0   = int(np.median(pos3))

        start_peak_id = sk if sk.startswith("5P_") else None
        end_peak_id   = ek if ek.startswith("3P_") else None

        if start_peak_id and end_peak_id:
            source = "peak+peak"
        elif start_peak_id and not end_peak_id:
            source = "peak+raw"
        elif (not start_peak_id) and end_peak_id:
            source = "raw+peak"
        else:
            source = "raw+raw"

        isoform_id = f"op{opid}__{sk}__{ek}__end{end_pos0}"

        rows.append({
            "chrom": chrom,
            "strand": strand,
            "merged_operon_id": opid,
            "isoform_id": isoform_id,
            "start_pos0": start_pos0,
            "end_pos0": end_pos0,
            "start_peak_id": start_peak_id,
            "end_peak_id": end_peak_id,
            "start_key": sk,
            "end_key": ek,
            "n_reads": n,
            "operon_total_reads_used": operon_reads_used,
            "frac_operon_reads": n / operon_reads_used,
            "start_spread_mad_bp": mad(pos5),
            "end_spread_mad_bp": mad(pos3),
            "source": source,
            "core_start0": core_s0,
            "core_end0": core_e0,
        })

        if write_membership:
            for p5, p3, qn in members:
                mem_rows.append({
                    "chrom": chrom,
                    "strand": strand,
                    "merged_operon_id": opid,
                    "isoform_id": isoform_id,
                    "read_id": qn,
                    "pos5p0": int(p5),
                    "pos3p0": int(p3),
                    "start_key": sk,
                    "end_key": ek,
                    "start_peak_id": start_peak_id,
                    "end_peak_id": end_peak_id,
                })

clusters = pd.DataFrame(rows)
if clusters.empty:
    print("No isoform clusters produced. Check filters / core columns.")
else:
    if DO_POST_MERGE:
        clusters = post_merge_isoforms(clusters, MERGE_END_TOL_BP)

    clusters = clusters.sort_values(
        ["chrom","strand","merged_operon_id","n_reads","start_pos0","end_pos0"],
        ascending=[True, True, True, False, True, True]
    ).reset_index(drop=True)

    clusters.to_csv(OUT_ISOFORMS_TSV, sep="\t", index=False)
    print(f"✅ wrote {OUT_ISOFORMS_TSV} | n_isoforms={len(clusters)}")

if write_membership and mem_rows:
    pd.DataFrame(mem_rows).to_csv(OUT_MEMBERSHIP_TSV, sep="\t", index=False)
    print(f"✅ wrote {OUT_MEMBERSHIP_TSV} | n_rows={len(mem_rows)}")

bam.close()


## Plot

In [ ]:
# ============================================================
# Publication-style operon plotter (v1.1) — requested changes
#
# Changes implemented:
# 1) Depth TSV is known format: 3 columns (chrom, pos, depth). Parser locked to that.
# 2) Gene arrows reflect GFF3 gene strand (NOT transcript direction), so antisense genes show opposite arrows.
#    - X-axis is still transcription-direction for the OPERON (5'→3' left->right).
# 3) Gene labels use gene name if present (Name=...), otherwise locus_tag only.
#    - Label format: "<Name>" if Name exists else "<locus_tag>"
#    - If you want "Name|locus_tag" instead, change format in make_gene_label().
# 4) Ends track option: plot 3' ends upside-down (negative) for visual separation.
#    - Toggle with FLIP_3P_ENDS = True/False
#
# Panels (top -> bottom):
# 1) Transcription-direction axis (operon-local coords)
# 2) Gene annotation (black arrows, gene-strand direction)
# 3) Sequencing depth on operon strand only
# 4) 5' (green, up) and 3' (red, down if enabled) ends on operon strand only
# 5) RNA isoforms on operon strand (thickness ~ count)
# ============================================================

from __future__ import annotations

import os
import math
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow
from matplotlib.ticker import FuncFormatter, FixedLocator
from matplotlib.patches import FancyArrowPatch

# ----------------------------
# Your paths (assumes you defined these earlier in notebook)
# ----------------------------
# REQUIRED globals (you provided earlier):
# MOTHER_FOLDER, ENDS_FILES, DEPTH_FILES, OPERONS_TSV, GFF3_FILE

# Isoform clusters file
ISOFOMR_FOLDER = MOTHER_FOLDER + "/isoform_clustering"
ISOFORMS_TSV   = ISOFOMR_FOLDER + "/isoform_clusters.tsv"

# Output
OUT_DIR = MOTHER_FOLDER + "/operon_plots_v1"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# Config knobs
# ----------------------------
PAD_BP = 200
PAD_BP_FRAC = 0.01
DEPTH_SMOOTH_WIN = 1  # odd looks nice; set 1 to disable

# Ends plotting mode:
#   "absolute" : raw end counts (bedGraph value)
#   "relative" : normalized by the max ends count in the plotted window (so peak max = 1)
ENDS_MODE = "absolute"   # or "relative"

# If ENDS_MODE == "absolute", you can optionally rescale for visibility (leave 1.0 normally)
ENDS_ABS_SCALE = 1.0

ENDS_Y_SCALE = 1.0
FLIP_3P_ENDS = False     # <--- NEW: 3' ends drawn downward if True

MIN_ISOFORM_FRAC = 0.005
MAX_ISOFORMS_TO_PLOT =100
ISOFORM_THICKNESS_SCALE = .5
ISOFORM_THICKNESS_MODE = "sqrt"  # "sqrt" | "linear" | "log1p"

# Gene track aesthetics
GENE_ARROW_THICKNESS = 0.28   # y thickness
GENE_ARROW_HEAD_MAX  = 60     # max head length in bp (tx coords)
GENE_ARROW_HEAD_MIN  = 10     # min head length
GENE_LABEL_FONTSIZE  = 12

LABEL_FONTSIZE = 15
# --- NEW: Syn3A GenBank file path ---
SYN3A_GB_FILE = GENOME_FOLDER + "/syn3a.gb"  # <-- CHANGE to your real path

def syn3a_locus_nums_from_genbank(gb_path: str) -> set[str]:
	"""
	Return locus_tag set from a Syn3A GenBank file.
	Requires: biopython installed (from Bio import SeqIO)
	"""
	from Bio import SeqIO

	tags: set[str] = set()
	for rec in SeqIO.parse(gb_path, "genbank"):
		for feat in rec.features:
			q = feat.qualifiers
			if "locus_tag" in q and len(q["locus_tag"]) > 0:
				locus_tag = str(q["locus_tag"][0])
				locus_num = locus_tag.split('_')[1]
				tags.add(locus_num)
	return tags

# Load Syn3A locus tags
syn3a_locus_nums = syn3a_locus_nums_from_genbank(SYN3A_GB_FILE)
print("Syn3A locus nums loaded:", len(syn3a_locus_nums))

# ----------------------------
# Utilities: coordinate transforms
# ----------------------------
@dataclass(frozen=True)
class OperonCoord:
	chrom: str
	strand: str
	opid: int
	start0: int
	end0: int

	def tx_of_genome_pos0(self, pos0: int) -> int:
		"""Map genomic pos0 -> operon-local transcript coordinate (0..len), left->right is operon transcription."""
		if self.strand == "+":
			return pos0 - self.start0
		else:
			return (self.end0 - pos0)

	@property
	def length(self) -> int:
		return self.end0 - self.start0


# ----------------------------
# Read operons / isoforms / genes
# ----------------------------
operons = pd.read_csv(OPERONS_TSV, sep="\t")
needed = {"chrom","strand","merged_operon_id","start0_final","end0_final","name"}
missing = needed - set(operons.columns)
if missing:
	raise ValueError(f"operons.final.tsv missing columns: {sorted(missing)}")

isoforms_all = pd.read_csv(ISOFORMS_TSV, sep="\t")
if "merged_operon_id" not in isoforms_all.columns:
	raise ValueError("ISOFORMS_TSV must contain merged_operon_id")


def read_genes_from_gff3(gff3_path: str, syn3a_locus_nums: set[str]) -> pd.DataFrame:
	"""
	Parse GFF3 (Syn1) and mark whether each gene exists in Syn3A.
	Keeps only 'gene' features.
	"""
	rows = []
	with open(gff3_path, "r") as f:
		for line in f:
			if not line or line.startswith("#"):
				continue
			parts = line.rstrip("\n").split("\t")
			if len(parts) != 9:
				continue
			chrom, source, ftype, start1, end1, score, strand, phase, attrs = parts
			if ftype != "gene":
				continue

			s1 = int(start1)
			e1 = int(end1)
			start0 = s1 - 1          # 1-based inclusive -> 0-based
			end0   = e1              # half-open end

			# parse attributes
			ad = {}
			for kv in attrs.split(";"):
				if "=" in kv:
					k, v = kv.split("=", 1)
					ad[k] = v

			locus = ad.get("locus_tag", "") or ""
			gene_name = ad.get("Name", "") or ""

			rows.append({
				"chrom": chrom,
				"start0": start0,
				"end0": end0,
				"strand": strand,
				"locus_tag": locus,
				"gene_name": gene_name,
				"in_syn3a": (locus.split('_')[1] in syn3a_locus_nums) if locus else False,
			})

	return pd.DataFrame(rows)

# Reload genes with Syn3A existence flag
genes = read_genes_from_gff3(GFF3_FILE, syn3a_locus_nums)
print("Syn1 genes parsed:", len(genes))
print("Syn1 genes present in Syn3A:", int(genes["in_syn3a"].sum()))


# ----------------------------
# Depth/Ends readers
# ----------------------------
def read_bedgraph(path: str) -> pd.DataFrame:
	df = pd.read_csv(path, sep="\t", header=None, comment="#")
	if df.shape[1] < 4:
		raise ValueError(f"bedGraph {path} has <4 columns")
	df = df.iloc[:, :4]
	df.columns = ["chrom","start0","end0","value"]
	df["start0"] = df["start0"].astype(int)
	df["end0"]   = df["end0"].astype(int)
	df["value"]  = pd.to_numeric(df["value"], errors="coerce").fillna(0.0)
	return df


def read_depth_tsv_3col(path: str) -> pd.DataFrame:
	"""
	LOCKED format per your note: 3 columns: chrom, pos, depth.
	pos is assumed 1-based if min(pos)==1; else assumed 0-based.
	Output normalized to: chrom, start0, end0, depth (per-base interval).
	"""
	df = pd.read_csv(path, sep="\t", header=None)
	if df.shape[1] < 3:
		raise ValueError(f"Depth TSV {path} expected 3 columns: chrom pos depth")
	df = df.iloc[:, :3]
	df.columns = ["chrom","pos","depth"]
	df["pos"] = df["pos"].astype(int)
	df["depth"] = pd.to_numeric(df["depth"], errors="coerce").fillna(0.0)

	pmin = int(df["pos"].min())
	if pmin == 1:
		start0 = df["pos"] - 1
	else:
		start0 = df["pos"]

	out = pd.DataFrame({
		"chrom": df["chrom"].astype(str),
		"start0": start0.astype(int),
		"end0": (start0 + 1).astype(int),
		"depth": df["depth"].astype(float),
	})
	return out


depth_plus  = read_depth_tsv_3col(DEPTH_FILES["plus"])
depth_minus = read_depth_tsv_3col(DEPTH_FILES["minus"])

ends_plus_5  = read_bedgraph(ENDS_FILES["plus_5"])
ends_plus_3  = read_bedgraph(ENDS_FILES["plus_3"])
ends_minus_5 = read_bedgraph(ENDS_FILES["minus_5"])
ends_minus_3 = read_bedgraph(ENDS_FILES["minus_3"])


# ----------------------------
# Subsetting helpers
# ----------------------------
def subset_intervals(df: pd.DataFrame, chrom: str, start0: int, end0: int,
					 start_col="start0", end_col="end0") -> pd.DataFrame:
	g = df[df["chrom"].astype(str) == str(chrom)]
	return g[(g[start_col] < end0) & (g[end_col] > start0)].copy()


def smooth_series(y: np.ndarray, win: int) -> np.ndarray:
	if win <= 1:
		return y
	if win % 2 == 0:
		win += 1
	k = np.ones(win, dtype=float) / win
	return np.convolve(y, k, mode="same")


def thickness_from_count(n: float) -> float:
	if ISOFORM_THICKNESS_MODE == "sqrt":
		return ISOFORM_THICKNESS_SCALE * math.sqrt(max(0.0, n))
	if ISOFORM_THICKNESS_MODE == "log1p":
		return ISOFORM_THICKNESS_SCALE * math.log1p(max(0.0, n))
	return ISOFORM_THICKNESS_SCALE * max(0.0, n)


def make_gene_label(locus_tag: str, gene_name: str) -> str:
	"""
	Requested: show locusNum/gene name, where gene name is from Name= in GFF3.
	Interpretation:
	  - If gene_name exists: use gene_name
	  - else: use locus_tag only
	If you want 'gene_name|locus_tag', change below.
	"""
	gene_name = (gene_name or "").strip()
	locus_tag = (locus_tag or "").strip()
	locus_num = locus_tag.split('_')[1]
	if gene_name:
		return locus_num + '/' + gene_name
	return locus_num

def genome_pos0_from_tx(oc: OperonCoord, x_tx: float) -> int:
	"""
	Convert operon-local tx coordinate (float) -> genomic pos0 (int).
	For + strand: genome = start0 + x
	For - strand: genome = end0 - x
	"""
	if oc.strand == "+":
		return int(round(oc.start0 + x_tx))
	else:
		return int(round(oc.end0 - x_tx))
	
def make_genome_formatter(oc: OperonCoord):
	def _fmt(x, pos=None):
		# x is tx coordinate
		if oc.strand == "+":
			g = oc.start0 + x
		else:
			g = oc.end0 - x
		return str(int(round(g)))
	return FuncFormatter(_fmt)


# ----------------------------
# Panel: Gene arrows (respect GFF3 gene strand)
# ----------------------------
def draw_gene_arrows(ax, oc: OperonCoord, genes_df: pd.DataFrame):
	"""
	Draw gene arrows reflecting GENE STRAND in genome, but on OPERON transcript-oriented x-axis.

	Axis definition: x increases left->right in OPERON transcription direction.
	Therefore, a gene's arrow direction depends on whether its strand matches the operon strand:
	  - If gene strand == operon strand: arrow points to the right
	  - Else: arrow points to the left
	"""
	if genes_df.empty:
		ax.text(0.01, 0.5, "No genes in interval", transform=ax.transAxes, va="center")
		ax.set_ylim(0, 1)
		ax.set_yticks([])
		ax.spines["left"].set_visible(False)
		ax.spines["right"].set_visible(False)
		ax.spines["bottom"].set_visible(False)
		return

	y = 0.5
	h = GENE_ARROW_THICKNESS

	# Sort by genomic start for stable labeling; x-placement is via tx mapping
	for _, r in genes_df.sort_values("start0").iterrows():
		g0 = int(r["start0"])
		g1 = int(r["end0"])
		gstrand = str(r["strand"])
		in_syn3a = bool(r.get("in_syn3a", False))

		x0 = oc.tx_of_genome_pos0(g0)
		x1 = oc.tx_of_genome_pos0(g1)
		left = min(x0, x1)
		right = max(x0, x1)
		width = max(1, right - left)

		# Determine arrow direction in plot space
		same_as_operon = (gstrand == oc.strand)
		head_len = min(GENE_ARROW_HEAD_MAX, max(GENE_ARROW_HEAD_MIN, int(0.2 * width)))

		if same_as_operon:
			# right-pointing
			arr = FancyArrow(left, y + h/2, width, 0,
							 width=0.1, head_width=h, head_length=head_len,
							 length_includes_head=True, color="black")
		else:
			# left-pointing: draw arrow from right -> left with negative dx
			arr = FancyArrow(right, y + h/2, -width, 0,
							 width=0.1, head_width=h, head_length=head_len,
							 length_includes_head=True, color="red")

		ax.add_patch(arr)

		label = make_gene_label(str(r.get("locus_tag","")), str(r.get("gene_name","")))
		# if label:
		#     ax.text((left + right)/2, y - 0.2, label,
		#             ha="center", va="top", fontsize=GENE_LABEL_FONTSIZE)
		if label:
			ax.text((left + right)/2, y - 0.2, label,
					ha="center", va="top", fontsize=GENE_LABEL_FONTSIZE, color="black")
			# If absent in Syn3A, add a small "×" marker next to label
			if not in_syn3a:
				ax.text((left + right)/2, y - 0.35, "×",
						ha="center", va="top", fontsize=GENE_LABEL_FONTSIZE+2, color="black")

	ax.set_ylim(0, 1)
	ax.set_yticks([])
	ax.spines["left"].set_visible(False)
	ax.spines["right"].set_visible(False)
	ax.spines["bottom"].set_visible(False)
	ax.tick_params(axis="x", which="both",
				bottom=False, top=False,      # no tick marks
				labelbottom=False)            # no tick labels

# ----------------------------
# Panel: Depth
# ----------------------------
def draw_depth(ax, oc: OperonCoord, depth_df: pd.DataFrame, plot_s0: int, plot_e0: int, strand: str):
	if depth_df.empty:
		ax.text(0.01, 0.5, "No depth", transform=ax.transAxes, va="center")
		return

	L = plot_e0 - plot_s0
	if L <= 0:
		return

	y = np.zeros(L, dtype=float)
	# depth_df here is per-base intervals; fill by overlap
	for _, r in depth_df.iterrows():
		a0 = max(plot_s0, int(r["start0"]))
		a1 = min(plot_e0, int(r["end0"]))
		if a1 <= a0:
			continue
		y[a0 - plot_s0 : a1 - plot_s0] = float(r["depth"])

	if DEPTH_SMOOTH_WIN > 1:
		y = smooth_series(y, DEPTH_SMOOTH_WIN)

	genome_positions = np.arange(plot_s0, plot_e0, dtype=int)
	x_tx = np.array([oc.tx_of_genome_pos0(int(p)) for p in genome_positions], dtype=int)

	order = np.argsort(x_tx)
	ax.plot(x_tx[order], y[order])
	strand_text = "Forward" if strand == "+" else "Reverse"
	ax.set_ylabel(f"Depth {strand_text} Strand", fontsize=LABEL_FONTSIZE)
	ax.spines["right"].set_visible(False)
	ax.spines["top"].set_visible(False)


# ----------------------------
# Panel: Ends (3' optionally flipped)
# ----------------------------

# ============================================================
# 1) TSV parsers (robust: auto-dtypes, bool handling)
# ============================================================

def read_peaks_5p_tsv(path: str) -> pd.DataFrame:
	df = pd.read_csv(path, sep="\t", dtype=str, na_filter=True)
	# numeric cols (add more if you want)
	num_cols = [
		"cluster_start0","cluster_end0","cluster_width","summit_pos0",
		"summit_count","cluster_total_count","n_reads","n_unique_pos",
		"pos_iqr","pos_entropy","clip_median","clip_mean",
		"clip_frac_ge_1","clip_frac_ge_5","clip_frac_ge_10",
		"clean_count_le_3","clean_count_le_5",
		"nearest_opposite_summit_pos0","nearest_opposite_dist",
		"summit_fraction","score_5p",
		"param_PEAK_HALF_WIN","param_LOCAL_MAX_RADIUS","param_MIN_SUMMIT_COUNT",
		"param_MIN_PEAK_DIST","param_MIN_PEAK_TOTAL","param_FRACTION_THRESHOLD",
		"minus10_6mer_shift","minus10_6mer_match","minus10_6mer_mm",
		"minus10_9mer_shift","minus10_9mer_match","minus10_9mer_mm",
	]
	for c in num_cols:
		if c in df.columns:
			df[c] = pd.to_numeric(df[c], errors="coerce")
	# standard cols
	for c in ["chrom","strand","end_type","peak_id","promoter_match_class"]:
		if c in df.columns:
			df[c] = df[c].astype(str)
	return df

def _to_bool_series(x: pd.Series) -> pd.Series:
	if x.dtype == bool:
		return x
	s = x.astype(str).str.strip().str.lower()
	return s.isin(["true","t","1","yes","y"])

def read_peaks_3p_tsv(path: str) -> pd.DataFrame:
	df = pd.read_csv(path, sep="\t", dtype=str, na_filter=True)
	num_cols = [
		"cluster_start0","cluster_end0","cluster_width","summit_pos0",
		"summit_count","cluster_total_count","n_reads","n_unique_pos",
		"pos_iqr","pos_entropy","clip_median","clip_mean",
		"clip_frac_ge_1","clip_frac_ge_5","clip_frac_ge_10",
		"clean_count_le_3","clean_count_le_5",
		"nearest_opposite_summit_pos0","nearest_opposite_dist",
		"u_run_max","u_frac_tail","hp_mfe_best","hp_mfe_best_rel_start",
		"btf_nearest_dist","btf_nearest_pos0","btf_nearest_score",
		"transterm_nearest_dist","transterm_nearest_pos0","transterm_nearest_score",
		"param_PEAK_HALF_WIN","param_LOCAL_MAX_RADIUS","param_MIN_SUMMIT_COUNT",
		"param_MIN_PEAK_DIST","param_MIN_PEAK_TOTAL","param_FRACTION_THRESHOLD",
	]
	for c in num_cols:
		if c in df.columns:
			df[c] = pd.to_numeric(df[c], errors="coerce")

	for b in ["btf_supported","transterm_supported","supported_any","supported_both"]:
		if b in df.columns:
			df[b] = _to_bool_series(df[b])

	for c in ["chrom","strand","end_type","peak_id","tail_seq_rna"]:
		if c in df.columns:
			df[c] = df[c].astype(str)
	return df

# ============================================================
# 2) Peak labelers for plotting overlays (no "none" labels)
# ============================================================

def add_5p_label(df5: pd.DataFrame,
				 label_col: str = "peak_label",
				 promoter_col: str = "promoter_match_class") -> pd.DataFrame:
	"""
	Label 5' peaks:
	  promoter_9mer -> '9mer'
	  promoter_6mer -> '6mer'
	  otherwise     -> ''   (no label)
	"""
	out = df5.copy()
	if promoter_col not in out.columns:
		out[label_col] = ""
		return out

	m = out[promoter_col].astype(str)
	out[label_col] = np.where(
		m.eq("promoter_9mer"), "9mer",
		np.where(m.eq("promoter_6mer"), "6mer", "")
	)
	return out

def add_3p_label(df3: pd.DataFrame,
				 label_col: str = "peak_label",
				 btf_col: str = "btf_supported",
				 tth_col: str = "transterm_supported") -> pd.DataFrame:
	"""
	Label 3' peaks:
	  btf=True,  tth=False -> 'btf'
	  btf=False, tth=True  -> 'tth'
	  btf=True,  tth=True  -> 'both'
	  neither             -> ''   (no label)
	"""
	out = df3.copy()
	if btf_col not in out.columns or tth_col not in out.columns:
		out[label_col] = ""
		return out

	btf = out[btf_col].fillna(False).astype(bool)
	tth = out[tth_col].fillna(False).astype(bool)

	out[label_col] = np.where(
		btf & tth, "both",
		np.where(btf, "btf", np.where(tth, "tth", ""))
	)
	return out

# ============================================================
# 3) Convenience: load + label in one shot
# ============================================================

def load_and_label_peaks(
	peaks5_tsv: str,
	peaks3_tsv: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
	df5 = add_5p_label(read_peaks_5p_tsv(peaks5_tsv), label_col="peak_label")
	df3 = add_3p_label(read_peaks_3p_tsv(peaks3_tsv), label_col="peak_label")
	# ensure summit_pos0 int for plotting
	for df in (df5, df3):
		if "summit_pos0" in df.columns:
			df["summit_pos0"] = pd.to_numeric(df["summit_pos0"], errors="coerce").astype("Int64")
	return df5, df3


from typing import Optional, Tuple

def draw_ends(
	ax,
	oc: OperonCoord,
	ends5_df: pd.DataFrame,
	ends3_df: pd.DataFrame,
	plot_s0: int,
	plot_e0: int,
	flip_3p: bool,
	mode: str = "absolute",          # "absolute" | "relative"
	abs_scale: float = 1.0,          # only used for absolute
	# NEW:
	peaks5_df: Optional[pd.DataFrame] = None,
	peaks3_df: Optional[pd.DataFrame] = None,
	peaks_pos_col: str = "summit_pos0",
	peaks_label_col_5p: Optional[str] = None,
	peaks_label_col_3p: Optional[str] = None,
	peaks_marker_lw: float = 1.6,
	peaks_text: bool = True,
	peaks_text_y_frac_5p: float = 0.92,  # text row in axes coords
	peaks_text_y_frac_3p: float = 0.08,
):
	def impulses(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
		if df.empty:
			return np.array([], dtype=int), np.array([], dtype=float)
		mid = ((df["start0"].astype(int) + df["end0"].astype(int)) // 2).astype(int)
		v = df["value"].astype(float).to_numpy()
		m = (mid >= plot_s0) & (mid < plot_e0)
		mid = mid[m]
		v = v[m]
		x = np.array([oc.tx_of_genome_pos0(int(p)) for p in mid], dtype=int)
		return x, v

	x5, y5 = impulses(ends5_df)
	x3, y3 = impulses(ends3_df)

	# --- scaling logic ---
	if mode not in {"absolute", "relative"}:
		raise ValueError("draw_ends: mode must be 'absolute' or 'relative'")

	if mode == "relative":
		if y5.size or y3.size:
			ymax = max(y5.max() if y5.size else 0.0, y3.max() if y3.size else 0.0)
			if ymax > 0:
				y5 = y5 / ymax
				y3 = y3 / ymax
		y5 = y5 * ENDS_Y_SCALE
		y3 = y3 * ENDS_Y_SCALE
		y_label = "Ends (rel.)"
	else:
		y5 = y5 * abs_scale
		y3 = y3 * abs_scale
		y_label = "Ends (count)"

	if flip_3p and y3.size:
		y3 = -y3

	# clear collections for clean redraw
	for coll in list(ax.collections):
		coll.remove()

	# raw impulses
	if x5.size:
		ax.vlines(x5, 0, y5, linewidth=2.0, color="green")
	if x3.size:
		ax.vlines(x3, 0, y3, linewidth=2.0, color="red")

	# labels
	ax.text(0.90, 0.85, "5' ends", transform=ax.transAxes, fontsize=LABEL_FONTSIZE, color="green")
	ax.text(0.90, 0.65, "3' ends", transform=ax.transAxes, fontsize=LABEL_FONTSIZE, color="red")

	ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
	ax.spines["right"].set_visible(False)
	ax.spines["top"].set_visible(False)
	if flip_3p:
		ax.axhline(0.0, linewidth=0.8)

	# ------------------------------------------------------------
	# NEW: overlay clustered peaks as thin marker lines + text ids
	# ------------------------------------------------------------
	def _infer_label_col(df: pd.DataFrame, explicit: Optional[str], candidates):
		if df is None:
			return None
		if explicit and explicit in df.columns:
			return explicit
		for c in candidates:
			if c in df.columns:
				return c
		return None

	peaks_require_label = True
	peaks_label_filter_col_5p = "peak_label"
	peaks_label_filter_col_3p = "peak_label"

	# --- 5' peaks overlay ---
	if peaks5_df is not None and len(peaks5_df):
		p5 = peaks5_df.copy()
		p5[peaks_pos_col] = p5[peaks_pos_col].astype(int)
		p5 = p5[(p5[peaks_pos_col] >= plot_s0) & (p5[peaks_pos_col] < plot_e0)]

		# filter out not-matched peaks (label == "")
		if peaks_require_label and peaks_label_filter_col_5p is not None and peaks_label_filter_col_5p in p5.columns:
			p5 = p5[p5[peaks_label_filter_col_5p].astype(str).ne("")]

		if not p5.empty:
			xpk = np.array([oc.tx_of_genome_pos0(int(p)) for p in p5[peaks_pos_col].to_numpy()], dtype=int)
			# ax.vlines(xpk, 0, ax.get_ylim()[1], linewidth=peaks_marker_lw, color="green", alpha=0.35)

			labcol = _infer_label_col(p5, peaks_label_col_5p, ["peak5_id", "peak_id", "cluster_id", "id", "peak_label"])
			if peaks_text and labcol is not None:
				for xp, lab in zip(xpk, p5[labcol].astype(str).to_list()):
					if lab == "":  # safety
						continue
					ax.text(
						xp, peaks_text_y_frac_5p, lab,
						transform=ax.get_xaxis_transform(),
						fontsize=LABEL_FONTSIZE-1, rotation=90, va="top", ha="center",
						color="green"
					)

	# --- 3' peaks overlay ---
	if peaks3_df is not None and len(peaks3_df):
		p3 = peaks3_df.copy()
		p3[peaks_pos_col] = p3[peaks_pos_col].astype(int)
		p3 = p3[(p3[peaks_pos_col] >= plot_s0) & (p3[peaks_pos_col] < plot_e0)]

		# filter out not-matched peaks (label == "")
		if peaks_require_label and peaks_label_filter_col_3p is not None and peaks_label_filter_col_3p in p3.columns:
			p3 = p3[p3[peaks_label_filter_col_3p].astype(str).ne("")]

		if not p3.empty:
			xpk = np.array([oc.tx_of_genome_pos0(int(p)) for p in p3[peaks_pos_col].to_numpy()], dtype=int)
			# ax.vlines(xpk, ax.get_ylim()[0], 0, linewidth=peaks_marker_lw, color="red", alpha=0.35)

			labcol = _infer_label_col(p3, peaks_label_col_3p, ["peak3_id", "peak_id", "cluster_id", "id", "peak_label"])
			if peaks_text and labcol is not None:
				for xp, lab in zip(xpk, p3[labcol].astype(str).to_list()):
					if lab == "":
						continue
					ax.text(
						xp, peaks_text_y_frac_3p, lab,
						transform=ax.get_xaxis_transform(),
						fontsize=LABEL_FONTSIZE-1, rotation=90, va="bottom", ha="center",
						color="red"
					)

ANNOTATED_5_PEAKS_TSV = "syn1_5p_ranked_with_minus10.tsv"
ANNOTAED_3_PEAKS_TSV = "syn1_3p_peaks_with_predictor_overlap.tsv"
peaks5_df, peaks3_df = load_and_label_peaks(ANNOTATED_5_PEAKS_TSV, ANNOTAED_3_PEAKS_TSV)

def subset_peaks(df: pd.DataFrame, chrom: str, strand: str, start0: int, end0: int,
				 pos_col: str = "summit_pos0") -> pd.DataFrame:
	g = df[(df["chrom"].astype(str) == str(chrom)) & (df["strand"].astype(str) == str(strand))].copy()
	g[pos_col] = g[pos_col].astype(int)
	return g[(g[pos_col] >= start0) & (g[pos_col] < end0)].copy()

# ----------------------------
# Panel: Isoforms
# ----------------------------
def _pack_intervals_min_rows(intervals: List[Tuple[int,int]]) -> List[int]:
	"""
	Given intervals [left,right] in TX coordinates, assign each to a row index so that
	no intervals overlap within a row. Greedy algorithm: sort by left, then place
	each into the first row whose current end <= left.
	Returns list of row indices aligned with input order.
	"""
	# sort by start while remembering original indices
	order = sorted(range(len(intervals)), key=lambda i: (intervals[i][0], intervals[i][1]))
	row_ends: List[int] = []   # current end for each row
	rows_out = [0] * len(intervals)

	for i in order:
		a, b = intervals[i]
		placed = False
		for r, end in enumerate(row_ends):
			if end <= a:  # no overlap (touching allowed)
				row_ends[r] = b
				rows_out[i] = r
				placed = True
				break
		if not placed:
			row_ends.append(b)
			rows_out[i] = len(row_ends) - 1

	return rows_out


def layout_isoform_tracks(iso_df: pd.DataFrame, oc: OperonCoord, plot_s0: int, plot_e0: int) -> pd.DataFrame:
	"""
	- Filter by MIN_ISOFORM_FRAC
	- Clip to plotting window
	- Group by start_pos0 (exact), then within each group:
		* sort by abundance
		* assign rows via interval packing to avoid overlap
	Adds columns:
	  tx_left, tx_right, group_id, y, lw, alpha
	"""
	if iso_df.empty:
		return iso_df

	iso = iso_df.copy()
	iso = iso[iso["frac_operon_reads"] >= MIN_ISOFORM_FRAC].copy()
	if iso.empty:
		return iso
	# print("before sorting:")
	# print(iso)
	if oc.strand == "+":
	# cap for safety (still respects your sort preferences)
		iso = iso.sort_values(["start_pos0","end_pos0","n_reads"], ascending=[True, True, False]).head(MAX_ISOFORMS_TO_PLOT).copy()
	else:
		print("Operon on strand -")
		iso = iso.sort_values(["start_pos0","end_pos0","n_reads"], ascending=[False, False, False]).head(MAX_ISOFORMS_TO_PLOT).copy()
	# print("after sorting and capping:")
	# print(iso)
	# clip in genome space then convert to tx intervals
	tx_lefts, tx_rights = [], []
	for _, r in iso.iterrows():
		s = int(r["start_pos0"])
		e = int(r["end_pos0"])
		s_clip = min(max(s, plot_s0), plot_e0)
		e_clip = min(max(e, plot_s0), plot_e0)
		x0 = oc.tx_of_genome_pos0(s_clip)
		x1 = oc.tx_of_genome_pos0(e_clip)
		tx_lefts.append(min(x0, x1))
		tx_rights.append(max(x0, x1))

	iso["tx_left"] = tx_lefts
	iso["tx_right"] = tx_rights

	# Drop zero-length after clipping (can happen if isoform is fully outside)
	iso = iso[iso["tx_right"] > iso["tx_left"]].copy()
	if iso.empty:
		return iso

	# Encode thickness & opacity within each start group
	def _scale_alpha(n, nmin, nmax):
		if nmax <= nmin:
			return 0.9
		# map to [0.5, 0.95]
		t = (n - nmin) / (nmax - nmin)
		return float(0.5 + 0.45 * t)

	ys = []
	lws = []
	alphas = []

	base_y = 0
	group_gap = 1  # vertical gap between start groups

		# Prepare style columns
	iso = iso.copy()
	iso["y"] = float("nan")
	iso["lw"] = float("nan")
	iso["alpha"] = float("nan")

	base_y = 0

	# Exact start_pos0 grouping
	for g_start, g in iso.groupby("start_pos0", sort=False):
		# IMPORTANT: sorting changes row order, but g.index still points to original rows in iso
		g = g.sort_values(["n_reads", "tx_right"], ascending=[False, True]).copy()

		# Row packing to avoid overlap (based on tx intervals)
		intervals = list(zip(g["tx_left"].astype(int).tolist(), g["tx_right"].astype(int).tolist()))
		rows = _pack_intervals_min_rows(intervals)  # rows[i] corresponds to g.iloc[i]

		# Thickness + opacity
		nvals = g["n_reads"].astype(float).to_numpy()
		nmin, nmax = float(nvals.min()), float(nvals.max())

		# Assign styles back to the right isoform rows using g.index
		iso.loc[g.index, "y"]     = [base_y + r for r in rows]
		iso.loc[g.index, "lw"]    = [max(0.3, thickness_from_count(float(n))) for n in nvals]
		iso.loc[g.index, "alpha"] = [_scale_alpha(float(n), nmin, nmax) for n in nvals]

		base_y += (max(rows) + 1) + group_gap

	# group id used for color mapping
	iso["group_id"] = iso["start_pos0"].astype(int)

	# Optional: if you need a compact 0..N-1 index later (NOT required for correctness)
	# iso = iso.reset_index(drop=True)

	# # Exact start_pos0 grouping
	# for g_start, g in iso.groupby("start_pos0", sort=False):
	#     g = g.sort_values(["n_reads","tx_right"], ascending=[False, True]).copy()

	#     # Row packing to avoid overlap (based on tx intervals)
	#     intervals = list(zip(g["tx_left"].astype(int).tolist(), g["tx_right"].astype(int).tolist()))
	#     rows = _pack_intervals_min_rows(intervals)

	#     # map to global y with group offset
	#     for r in rows:
	#         ys.append(base_y + r)

	#     # thickness from your function
	#     for n in g["n_reads"].astype(float).to_numpy():
	#         lws.append(max(0.3, thickness_from_count(float(n))))

	#     nvals = g["n_reads"].astype(float).to_numpy()
	#     nmin, nmax = float(nvals.min()), float(nvals.max())
	#     for n in nvals:
	#         alphas.append(_scale_alpha(float(n), nmin, nmax))

	#     base_y += (max(rows) + 1) + group_gap

	# # Careful: we appended in group iteration order, but need to assign back in same row order of iso
	# # Because we iterated groupby which preserves row order within each group dataframe,
	# # we can just assign sequentially.
	# iso = iso.reset_index(drop=True)
	# iso["y"] = ys
	# iso["lw"] = lws
	# iso["alpha"] = alphas

	# # group id used for color mapping
	# iso["group_id"] = iso["start_pos0"].astype(int)

	return iso



def draw_isoforms(ax, oc: OperonCoord, iso_df: pd.DataFrame, plot_s0: int, plot_e0: int):
	if iso_df.empty:
		ax.text(0.01, 0.5, "No isoforms", transform=ax.transAxes, va="center")
		ax.set_yticks([])
		return

	iso = layout_isoform_tracks(iso_df, oc, plot_s0, plot_e0)
	# print(iso)
	if iso.empty:
		ax.text(0.01, 0.5, "No isoforms (after filter)", transform=ax.transAxes, va="center")
		ax.set_yticks([])
		return

	# --- Color mapping: same start_pos0 -> same color ---
	# Use a qualitative colormap and stable indexing by sorted unique starts.
	starts = sorted(iso["group_id"].unique().tolist())
	cmap = plt.get_cmap("tab10")  # good for categorical groups (no explicit colors hard-coded)
	color_map = {s: cmap(i % cmap.N) for i, s in enumerate(starts)}

	# Arrow aesthetics
	arrow_thickness = 0.0  # FancyArrow "width" is weird for horizontal arrows; use 0.0 and head_width only
	head_width = 0.2
	head_max = 35
	head_min = 8

	# Draw each isoform as an arrow (always pointing right in TX space)
	for _, r in iso.iterrows():
		left = float(r["tx_left"])
		right = float(r["tx_right"])
		y = float(r["y"])
		lw = float(r["lw"])
		alpha = float(r["alpha"])
		col = color_map[int(r["group_id"])]

		width = max(1.0, right - left)
		head_len = min(head_max, max(head_min, 0.18 * width))

		# Use FancyArrow; "width" is the tail thickness in data y-units, not linewidth.
		# We'll encode thickness via linewidth instead (more intuitive),
		# and keep arrow body "width" minimal.
		arr = FancyArrowPatch(
			(left, y), (right, y),
			arrowstyle='-|>',                 # triangular head with a bar
			# mutation_scale=head_len,          # controls head size (roughly in points)
			linewidth=lw,
			color=col,
			alpha=alpha,
			shrinkA=0, shrinkB=0,             # DO NOT shrink ends; keep exact endpoints
			# capstyle="butt",
			# joinstyle="miter",
		)
		
		# arr = FancyArrow(
		#     left, y, width, 0,
		#     width=0.0001,                     # near-zero body; thickness comes from linewidth below
		#     head_width=head_width,
		#     head_length=head_len,
		#     length_includes_head=True,
		#     facecolor=col,
		#     edgecolor=col,
		#     alpha=alpha,
		#     linewidth=lw,
		# )
		ax.add_patch(arr)

	# Tight y-limits (dynamic spacing already handled by packing)
	y_max = float(iso["y"].max()) if len(iso) else 1.0
	ax.set_ylim(-1, y_max + 1)

	ax.set_ylabel("Isoforms", fontsize=LABEL_FONTSIZE)
	ax.spines["right"].set_visible(False)
	ax.spines["top"].set_visible(False)
	ax.set_yticks([])


def filter_genes_for_plot(
	genes_df: pd.DataFrame,
	chrom: str,
	plot_s0: int,
	plot_e0: int,
	gene_subset_locus: Optional[List[str]] = None,
	gene_subset_name: Optional[List[str]] = None,
	gene_subset_span: Optional[Tuple[int, int]] = None,
) -> pd.DataFrame:
	"""
	Filter genes to:
	  (1) overlap the current plotting window [plot_s0, plot_e0)
	  (2) optionally overlap gene_subset_span
	  (3) optionally match locus_tag and/or gene_name lists
	"""
	g = genes_df[(genes_df["chrom"].astype(str) == str(chrom)) &
				 (genes_df["start0"] < plot_e0) & (genes_df["end0"] > plot_s0)].copy()

	if gene_subset_span is not None:
		a0, a1 = int(gene_subset_span[0]), int(gene_subset_span[1])
		g = g[(g["start0"] < a1) & (g["end0"] > a0)].copy()

	if gene_subset_locus is not None:
		keep = set(map(str, gene_subset_locus))
		g = g[g["locus_tag"].astype(str).isin(keep)].copy()

	if gene_subset_name is not None:
		keep = set(map(str, gene_subset_name))
		g = g[g["gene_name"].astype(str).isin(keep)].copy()

	return g

def get_xticklabels(left: int, right: int, strand: str):
	span = abs(right - left)

	if span <= 2000:
		step = 200
	elif span <= 5000:
		step = 500
	elif span <= 15000:
		step = 1000
	else:
		step = 2000

	small = min(left, right)
	large = max(left, right)

	# Round outward to multiples of step
	start = (small // step) * step
	if start > small:
		start -= step

	end = ((large + step - 1) // step) * step

	ticks = np.arange(start, end + step, step, dtype=int)

	# Return in transcription direction
	if strand == "+":
		return ticks
	else:
		return ticks[::-1]


def plot_one_operon(
	merged_operon_id: int,
	save_path: str,
	dpi: int = 300,
	gene_subset_locus: Optional[List[str]] = None,
	gene_subset_name: Optional[List[str]] = None,
	gene_subset_span: Optional[Tuple[int, int]] = None,
	crop_to_gene_subset: bool = False,
):
	op = operons[operons["merged_operon_id"] == merged_operon_id]
	if op.empty:
		raise ValueError(f"merged_operon_id {merged_operon_id} not found.")
	op = op.iloc[0]

	chrom  = str(op["chrom"])
	strand = str(op["strand"])
	s0 = int(op["start0_final"])
	e0 = int(op["end0_final"])
	oc = OperonCoord(chrom=chrom, strand=strand, opid=int(merged_operon_id), start0=s0, end0=e0)

	# Select strand-specific tracks for the OPERON strand
	depth_df = depth_plus if strand == "+" else depth_minus
	ends5_df = ends_plus_5 if strand == "+" else ends_minus_5
	ends3_df = ends_plus_3 if strand == "+" else ends_minus_3

	# Default plotting window from operon final bounds
	plot_s0 = s0 - int(PAD_BP_FRAC * (e0 - s0))
	plot_e0 = e0 + int(PAD_BP_FRAC * (e0 - s0))

	# ---- genes (optionally subset) ----
	genes_sub = pd.DataFrame()
	if not genes.empty:
		genes_sub = filter_genes_for_plot(
			genes_df=genes,
			chrom=chrom,
			plot_s0=plot_s0,
			plot_e0=plot_e0,
			gene_subset_locus=gene_subset_locus,
			gene_subset_name=gene_subset_name,
			gene_subset_span=gene_subset_span,
		)

	# ---- optional crop to selected genes ----
	# If enabled and genes_sub is non-empty, redefine plot window to cover union of selected genes (+/- PAD_BP)
	if crop_to_gene_subset and (genes_sub is not None) and (not genes_sub.empty):
		gmin = int(genes_sub["start0"].min())
		gmax = int(genes_sub["end0"].max())
		plot_s0 = gmin - int(PAD_BP_FRAC * (gmax - gmin))
		plot_e0 = gmax + int(PAD_BP_FRAC * (gmax - gmin))

		# Re-filter genes to the new window (important if you also used gene_subset_span)
		genes_sub = filter_genes_for_plot(
			genes_df=genes,
			chrom=chrom,
			plot_s0=plot_s0,
			plot_e0=plot_e0,
			gene_subset_locus=gene_subset_locus,
			gene_subset_name=gene_subset_name,
			gene_subset_span=gene_subset_span,
		)

	# ---- subset depth/ends to (possibly updated) plot window ----
	depth_sub = subset_intervals(depth_df, chrom, plot_s0, plot_e0, start_col="start0", end_col="end0")
	ends5_sub = subset_intervals(ends5_df, chrom, plot_s0, plot_e0, start_col="start0", end_col="end0")
	ends3_sub = subset_intervals(ends3_df, chrom, plot_s0, plot_e0, start_col="start0", end_col="end0")

	# ---- isoforms (still by operon_id; x-range clipping happens inside draw_isoforms) ----
	iso_sub = isoforms_all[isoforms_all["merged_operon_id"] == merged_operon_id].copy()
	for col in ["start_pos0","end_pos0","n_reads","frac_operon_reads"]:
		if col not in iso_sub.columns:
			raise ValueError(f"ISOFORMS_TSV missing required column: {col}")

	# ---- Build figure ----
	fig = plt.figure(figsize=(24, 14))
	gs = fig.add_gridspec(5, 1, height_ratios=[0.5, 1.2, 2.0, 1.5, 2.2], hspace=0.25)

	ax0 = fig.add_subplot(gs[0, 0])                  # axis
	ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)      # genes
	ax2 = fig.add_subplot(gs[2, 0], sharex=ax0)      # depth
	ax3 = fig.add_subplot(gs[3, 0], sharex=ax0)      # ends
	ax4 = fig.add_subplot(gs[4, 0], sharex=ax0)      # isoforms

	# ---- Panel 1: transcription-direction axis ----
	# Determine x-limits for padded window (always increasing on plot)
	x_left  = oc.tx_of_genome_pos0(plot_s0 if strand == "+" else plot_e0)
	x_right = oc.tx_of_genome_pos0(plot_e0 if strand == "+" else plot_s0)
	left, right = min(x_left, x_right), max(x_left, x_right)
	ax0.set_xlim(left, right)
	# Shared x is TX coordinate. We'll keep that shared formatter for ax4.
	# For ax0, create a *separate* top axis that labels TX ticks in GENOME coords.

	ticks_tx = get_xticklabels(left, right, strand="+")  # tx tick positions (ascending)

	# Hide any bottom ticks/labels on ax0 itself (optional but cleaner)
	ax0.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

	# Create an independent top x-axis for genomic labels
	ax0_top = ax0.twiny()
	ax0_top.set_xlim(ax0.get_xlim())  # must match tx limits

	ax0_top.xaxis.set_major_locator(FixedLocator(ticks_tx))
	ax0_top.xaxis.set_major_formatter(make_genome_formatter(oc))
	ax0_top.set_xlabel("Genomic position (0-based)", fontsize=11)

	# Make sure only TOP ticks/labels show
	ax0_top.tick_params(axis="x", which="both", top=True, labeltop=True, bottom=False, labelbottom=False)

	ax0.set_ylim(0, 1)
	ax0.set_yticks([])
	strand_text = "forward (+)" if strand == "+" else "reverse (-)"
	title = f"Operon {merged_operon_id} | {chrom} {strand_text} | genome [{s0}, {e0}) | name={op.get('name','')}"
	if crop_to_gene_subset and (genes_sub is not None) and (not genes_sub.empty):
		title += " | cropped-to-gene-subset"
	ax0.set_title(title, fontsize=11, loc="center")

	# shaded operon body (excluding padding)
	op_left  = min(oc.tx_of_genome_pos0(s0), oc.tx_of_genome_pos0(e0))
	op_right = max(oc.tx_of_genome_pos0(s0), oc.tx_of_genome_pos0(e0))
	ax0.axvspan(op_left, op_right, alpha=0.15)

	# ---- Panel 2: genes (gene-strand arrows) ----
	draw_gene_arrows(ax1, oc, genes_sub)

	# ---- Panel 3: depth ----
	draw_depth(ax2, oc, depth_sub, plot_s0, plot_e0, strand)

	# ---- Panel 4: ends (3' optionally flipped) ----
	peaks5_sub = subset_peaks(peaks5_df, chrom, oc.strand, plot_s0, plot_e0, pos_col="summit_pos0")
	peaks3_sub = subset_peaks(peaks3_df, chrom, oc.strand, plot_s0, plot_e0, pos_col="summit_pos0")

	draw_ends(
		ax3, oc, ends5_sub, ends3_sub,
		plot_s0, plot_e0,
		flip_3p=FLIP_3P_ENDS,
		mode=ENDS_MODE,
		abs_scale=ENDS_ABS_SCALE,
		peaks5_df=peaks5_sub,
		peaks3_df=peaks3_sub,
		peaks_pos_col="summit_pos0",
		peaks_label_col_5p="peak_label",  # <- uses 9mer/6mer
		peaks_label_col_3p="peak_label",  # <- uses btf/tth/both
		peaks_text=True,
	)


	# ---- Panel 5: isoforms ----
	draw_isoforms(ax4, oc, iso_sub, plot_s0, plot_e0)

	# Hide x tick labels except bottom panel
	for ax in [ax1, ax2, ax3]:
		plt.setp(ax.get_xticklabels(), visible=False)

	ax4.xaxis.set_major_locator(FixedLocator(ticks_tx))
	ax4.xaxis.set_major_formatter(FuncFormatter(lambda x, pos=None: str(int(round(x)))))
	ax4.set_xlabel("Transcript coordinate (bp, 5'→3')", fontsize=15)

	# ax4.set_xticks(ticks_tx)
	# ax4.set_xticklabels([str(int(t)) for t in ticks_tx], fontsize=10)
	fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
	plt.close(fig)





Syn3A locus nums loaded: 496
Syn1 genes parsed: 911
Syn1 genes present in Syn3A: 493


395 ATP synthase
91 HupA
366 rRNA

In [36]:

# ----------------------------
# Example usage (single operon)
# ----------------------------
# OPERON_ROW = 395
# gene_locusNums = np.arange(797, 788,-1).astype(str).tolist()  # example locus tags to highlight
# gene_subset = [f"MMSYN1_0{n}" for n in gene_locusNums]
# TEST_OPERON_ID = int(operons.iloc[OPERON_ROW]["merged_operon_id"])
# out_path = os.path.join(OUT_DIR, f"operon_{TEST_OPERON_ID}_subset.pdf")

OPERON_ROW = 366
# gene_locusNums = np.arange(797, 788,-1).astype(str).tolist()  # example locus tags to highlight
gene_subset = None
TEST_OPERON_ID = int(operons.iloc[OPERON_ROW]["merged_operon_id"])
out_path = os.path.join(OUT_DIR, f"operon_{TEST_OPERON_ID}.pdf")


plot_one_operon(TEST_OPERON_ID, 
				out_path,
				dpi=200,
				gene_subset_locus=gene_subset,
				crop_to_gene_subset=True)

print("Wrote:", out_path)

# ----------------------------
# Batch rendering (optional)
# ----------------------------
# for opid in operons["merged_operon_id"].astype(int).tolist():
#     save_path = os.path.join(OUT_DIR, f"operon_{opid}.pdf")
#     plot_one_operon(int(opid), save_path)
# print("Done.")

Operon on strand -
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_377.pdf


In [37]:
for ROW in range(len(operons)):
	OPERON_ROW = ROW
	# gene_locusNums = np.arange(797, 788,-1).astype(str).tolist()  # example locus tags to highlight
	gene_subset = None
	TEST_OPERON_ID = int(operons.iloc[OPERON_ROW]["merged_operon_id"])
	out_path = os.path.join(OUT_DIR, f"operon_{TEST_OPERON_ID}.pdf")


	plot_one_operon(TEST_OPERON_ID, 
					out_path,
					dpi=200,
					gene_subset_locus=gene_subset,
					crop_to_gene_subset=False)

	print("Wrote:", out_path)

Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_0.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_1.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_3.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_4.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_5.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_6.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_7.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_8.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_9.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_analysis/operon_plots_v1/operon_10.pdf
Wrote: /data/enguang/CMEODE/TRSC/Transcriptomics/RNAseq_ana

In [45]:
operons

,chrom,strand,merged_operon_id,start0_merged,end0_merged,start0_polished,end0_polished,n_reads_assigned,polish_method,q5p,...,member_operon_ids,merge_rule,name,locus_tags_raw,locus_tags,start0_final,end0_final,expanded_to_genes,gene_related,drop_reason
0,CP002027.1,+,0,0,2659,0,2659,13624,quantile,0.02,...,0,no_merge,mop0,"MMSYN1_0001,MMSYN1_0001,MMSYN1_0002,MMSYN1_0002","MMSYN1_0001,MMSYN1_0002",0,2659,False,True,NaN
1,CP002027.1,+,1,2659,5255,2659,5255,497,quantile,0.02,...,"1,2",boundary_within_same_strand_gene,mop1,"MMSYN1_0003,MMSYN1_0003,MMSYN1_0004,MMSYN1_000...","MMSYN1_0003,MMSYN1_0004,MMSYN1_0005",2659,5255,False,True,NaN
2,CP002027.1,+,3,5465,9979,5465,9978,5551,quantile,0.02,...,4,no_merge,mop3,"MMSYN1_0006,MMSYN1_0006,MMSYN1_0007,MMSYN1_0007","MMSYN1_0006,MMSYN1_0007",5465,9978,False,True,NaN
3,CP002027.1,+,4,18313,27518,18318,27518,4250,quantile,0.02,...,"5,6",boundary_within_same_strand_gene,mop4,"MMSYN1_0013,MMSYN1_0013,MMSYN1_0014,MMSYN1_001...","MMSYN1_0013,MMSYN1_0014,MMSYN1_0015,MMSYN1_001...",18318,27529,True,True,NaN
4,CP002027.1,+,5,27518,28635,27522,28635,32138,quantile,0.02,...,7,no_merge,mop5,"MMSYN1_0918,MMSYN1_0918",MMSYN1_0918,27522,28635,False,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,CP002027.1,-,433,1048753,1053270,1048881,1053378,177,quantile,0.02,...,"276,275",boundary_within_same_strand_gene,mop433,"MMSYN1_0892,MMSYN1_0892,MMSYN1_0891,MMSYN1_089...","MMSYN1_0892,MMSYN1_0891,MMSYN1_0890,MMSYN1_088...",1048867,1054231,True,True,NaN
422,CP002027.1,-,434,1054769,1065182,1054769,1065054,3382,core_fallback_shrink_guard,0.02,...,"274,273",boundary_within_same_strand_gene,mop434,"MMSYN1_0897,MMSYN1_0897,MMSYN1_0896,MMSYN1_089...","MMSYN1_0897,MMSYN1_0896,MMSYN1_0895,MMSYN1_089...",1054361,1065054,True,True,NaN
423,CP002027.1,-,435,1065182,1069768,1065182,1069767,3609,quantile,0.02,...,"272,271",boundary_within_same_strand_gene,mop435,"MMSYN1_0899,MMSYN1_0899,MMSYN1_0898,MMSYN1_0898","MMSYN1_0899,MMSYN1_0898",1065182,1069767,False,True,NaN
424,CP002027.1,-,436,1075089,1076808,1075089,1076784,426,quantile,0.02,...,270,no_merge,mop436,"MMSYN1_0907,MMSYN1_0907,MMSYN1_0906,MMSYN1_0906","MMSYN1_0907,MMSYN1_0906",1075089,1076784,False,True,NaN


### Plot Potential RNaseIII cleaveged operons

18 gene homology in syn1 found from RNase III cleavages in B. subtilis. 

In [62]:
from pathlib import Path

RNase_III_GENES_FOLDER = Path("./map_RNase_Bsubtilis/rnaseIII_syn1_operons")

if not RNase_III_GENES_FOLDER.exists():
	RNase_III_GENES_FOLDER.mkdir()

CANDIDATE_TSV = Path("./map_RNase_Bsubtilis/rnaseIII_genes_transferred_to_syn1_with_coords.tsv")

cand = pd.read_csv(CANDIDATE_TSV, sep="\t")
cand = cand.dropna(subset=["syn1_locus_tag"])  # focus on candidates with locus tags (for better annotation and merging)

cand_locus_tags = set(cand["syn1_locus_tag"])
print("Candidate locus tags in Syn1:")
print(cand_locus_tags)

gene_subset = None

for index, row in operons.iterrows():
	# gene_locusNums = np.arange(797, 788,-1).astype(str).tolist()  # example locus tags to highlight
	OPERON_ID = int(row["merged_operon_id"])
	covered_locus_tags = row["locus_tags"].split(",") if pd.notna(row["locus_tags"]) else []
	intersection = set(covered_locus_tags).intersection(cand_locus_tags)
	if not intersection:
		continue  # skip operons that don't cover any candidate genes
	out_path = os.path.join(RNase_III_GENES_FOLDER, f"operon_{OPERON_ID}_{'_'.join(intersection)}.pdf")

	plot_one_operon(OPERON_ID, 
					out_path,
					dpi=300,
					gene_subset_locus=gene_subset,
					crop_to_gene_subset=False)

	print("Wrote:", out_path)

Candidate locus tags in Syn1:
{'MMSYN1_0914', 'MMSYN1_0524', 'MMSYN1_0418', 'MMSYN1_0645', 'MMSYN1_0651', 'MMSYN1_0157', 'MMSYN1_0022', 'MMSYN1_0643', 'MMSYN1_0150', 'MMSYN1_0023', 'MMSYN1_0003', 'MMSYN1_0226', 'MMSYN1_0537', 'MMSYN1_0652', 'MMSYN1_0545', 'MMSYN1_0394', 'MMSYN1_0304', 'MMSYN1_0792'}
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_1_MMSYN1_0003.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_4_MMSYN1_0914.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_9_MMSYN1_0022_MMSYN1_0023.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_39_MMSYN1_0150.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_45_MMSYN1_0157.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_67_MMSYN1_0226.pdf
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_104_MMSYN1_0394.pdf
Operon on strand -
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/operon_224_MMSYN1_0914.pdf
Operon on strand -
Wrote: map_RNase_Bsubtilis/rnaseIII_syn1_operons/o

In [44]:
cand

,primary_name,primary_locus_tag,strand,n_candidate_sites,site_positions,syn1_gene,syn1_locus_tag,syn1_product,pident,bsub_cov_pct,...,has_syn1_homolog,syn1_seqid,syn1_start_1b,syn1_end_1b,syn1_strand,syn1_primary_id_y,syn1_protein_id,syn1_gene_annot,syn1_product_annot,syn1_primary_id
0,ecfA/BSU_01450/BSU01450,BSU_01450,+,2,"150762,150858",NaN,MMSYN1_0643,cobalt import ATP-binding protein CbiO 2,42.520,90.391459,...,True,CP002027.1,789817.0,791043.0,-,MMSYN1_0643,ADH21506.1,NaN,cobalt import ATP-binding protein CbiO 2,NaN
1,rsmH/BSU_15140/BSU15140,BSU_15140,+,2,"1581006,1581170",mraW,MMSYN1_0524,S-adenosyl-methyltransferase MraW,48.077,100.321543,...,True,CP002027.1,626485.0,627411.0,-,MMSYN1_0524,ADH21524.1,mraW,S-adenosyl-methyltransferase MraW,NaN
2,fabG/BSU_15910/BSU15910,BSU_15910,+,2,"1664856,1664911",NaN,MMSYN1_0022,sorbitol-6-phosphate 2-dehydrogenase (Glucitol...,32.941,103.658537,...,True,CP002027.1,38921.0,39679.0,-,MMSYN1_0022,ADH22201.1,NaN,sorbitol-6-phosphate 2-dehydrogenase (Glucitol...,NaN
3,mgtE/BSU_13300/BSU13300,BSU_13300,+,2,"1397097,1397124",mgtE,MMSYN1_0157,magnesium transporter,27.130,98.891353,...,True,CP002027.1,202331.0,203734.0,-,MMSYN1_0157,ADH21779.1,mgtE,magnesium transporter,NaN
4,atpA/BSU_36830/BSU36830,BSU_36830,-,2,"3784452,3784559",atpA,MMSYN1_0792,"ATP synthase F1, alpha subunit",60.254,94.223108,...,True,CP002027.1,932169.0,933746.0,-,MMSYN1_0792,ADH21457.1,atpA,"ATP synthase F1, alpha subunit",NaN
5,adk/BSU_01370/BSU01370,BSU_01370,+,1,146127,adk,MMSYN1_0651,adenylate kinase,38.710,85.714286,...,True,CP002027.1,794477.0,795118.0,-,MMSYN1_0651,ADH21786.1,adk,adenylate kinase,NaN
6,fusA/BSU_01120/BSU01120,BSU_01120,+,1,132069,fusA,MMSYN1_0150,translation elongation factor G,71.843,99.566474,...,True,CP002027.1,189950.0,192019.0,+,MMSYN1_0150,ADH21589.1,fusA,translation elongation factor G,NaN
7,rpoA/BSU_01430/BSU01430,BSU_01430,+,1,149329,rpoA,MMSYN1_0645,"DNA-directed RNA polymerase, alpha subunit",45.141,101.592357,...,True,CP002027.1,791530.0,792483.0,-,MMSYN1_0645,ADH22088.1,rpoA,"DNA-directed RNA polymerase, alpha subunit",NaN
8,lonA/BSU_28200/BSU28200,BSU_28200,-,1,2882649,lon,MMSYN1_0394,ATP-dependent protease La,46.382,100.000000,...,True,CP002027.1,480950.0,483310.0,+,MMSYN1_0394,ADH22086.1,lon,ATP-dependent protease La,NaN
9,cdsA/BSU_16540/BSU16540,BSU_16540,+,1,1722692,cdsA,MMSYN1_0304,phosphatidate cytidylyltransferase,33.679,71.747212,...,True,CP002027.1,387049.0,388077.0,-,MMSYN1_0304,ADH21783.1,cdsA,phosphatidate cytidylyltransferase,NaN


In [46]:
a = set([1,2])

In [49]:
b = set([2,3,1])

print(a.intersection(b))

{1, 2}


In [51]:
c = a.intersection(b)

print(c)

{1, 2}
